# RQ2 — Development-only KPI availability preflight
**Run only after PyCharm unit tests pass.** This is NOT an LLM experiment. It scans daily node/container telemetry CSV members but aggregates rows only within SHA-verified development windows; CSV members may physically contain other-split rows. Never input these results to a model or use OOD test telemetry to refine the catalog.


In [ ]:
# RQ2 — KPI PRESENCE ON SHA-VERIFIED DEVELOPMENT WINDOWS ONLY.
# No model calls or writes. Daily CSV streaming may decode rows from other
# splits; ONLY development-window rows are admitted to aggregates.
from google.colab import drive
from pathlib import Path
import json
import os
import subprocess
import sys

BRANCH = "feature/ood-knowledge-transfer-audit-2026-11-10"
REPO = Path("/content/Tesi_OpenRCA_OOD_Audit")
ROOT = Path("/content/drive/MyDrive/Tesi_OpenRCA")
URL = "https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git"

drive.mount("/content/drive")
for rel in ("Market.zip", "data/processed/split_market_v1.csv",
            "data/processed/benchmark_inputs_market.csv"):
    if not (ROOT / rel).is_file():
        raise FileNotFoundError("Required original GT-free source is absent: " + rel)

if REPO.exists():
    if not (REPO / ".git").is_dir():
        raise RuntimeError("Unexpected pre-existing path; won't overwrite or delete it")
    branch_now = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"], text=True
    ).strip()
    if branch_now != BRANCH:
        raise RuntimeError("Unexpected branch; no automatic checkout")
    if subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True
    ).strip():
        raise RuntimeError("Local changes; no automatic pull or overwrite")
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True, capture_output=True, text=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch",
         BRANCH, URL, str(REPO)],
        check=True, capture_output=True, text=True,
    )
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
env = dict(os.environ)
env["PYTHONPATH"] = str(REPO / "src")

result = subprocess.run(
    [sys.executable, "-m", "evaluation.rq2_development_kpi_audit",
     "--data-dir", str(ROOT)],
    cwd=REPO, env=env, capture_output=True, text=True,
)
if result.returncode:
    raise RuntimeError("Development KPI preflight failed:\n" + result.stderr[-8000:])
try:
    report = json.loads(result.stdout)
except json.JSONDecodeError as exc:
    raise RuntimeError("No valid development KPI audit summary") from exc
if not (
    report.get("audit") == "rq2_development_kpi_presence_audit_v0"
    and report.get("status") == "DEVELOPMENT_ONLY_GT_FREE_NUMERIC_PRESENCE"
    and report.get("source_verified") is True
    and report.get("development_cases") == 92
    and report.get("statistics_only_development_windows") is True
    and report.get("read_ground_truth") is False
    and report.get("read_record_csv") is False
    and report.get("read_scoring") is False
    and report.get("read_ood_telemetry_for_analysis") is False
    and report.get("possible_non_development_rows_decoded_from_shared_csv") is True
    and report.get("read_model_predictions") is False
    and report.get("model_api_calls") == 0
    and report.get("files_written") is False
    and report.get("can_execute_rule") is False
    and report.get("catalog_physically_verified") is False
):
    raise RuntimeError("Audit violated the blocked GT-free development protocol")

print("=== RQ2 — KPI AVAILABILITY ON DEVELOPMENT ONLY ===")
print("Branch:", BRANCH)
print("Commit:", commit)
print(json.dumps(report, indent=2, ensure_ascii=False))
print("\nNO API, NO GT, NO SCORING, NO FILE WRITES.")
print("ONLY DEVELOPMENT-WINDOW ROWS COUNTED. PHYSICAL UNITS NOT VERIFIED.")
